In [8]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import optuna
import lightgbm as lgb

In [2]:
df = pd.read_parquet('BTC_1h_logRV_ML_dataset.parquet')
df.head()

,log_rv_lag_1h,log_rv_lag_2h,log_rv_lag_3h,log_rv_lag_6h,log_rv_lag_12h,log_rv_lag_24h,log_rv_lag_48h,log_rv_lag_72h,log_rv_lag_168h,log_rv_mean_3h,...,is_us_session,is_eu_us_overlap,is_low_vol_168h,is_high_vol_168h,log_rv_lag1_x_us_session,log_rv_lag1_x_weekend,log_volume_lag1_x_us_session,abs_ret_lag1_x_us_session,target_log_rv,target_rv
datetime,,,,,,,,,,,,,,,,,,,,,
2020-04-24 10:00:00+00:00,-11.037855,-10.484218,-10.835839,-10.888887,-10.680269,-11.994565,-12.063372,-10.541077,-11.278820,-10.785971,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-10.399756,0.000030
2020-04-24 11:00:00+00:00,-10.399756,-11.037855,-10.484218,-11.590795,-9.354094,-11.090701,-11.917092,-9.388464,-11.179881,-10.640610,...,0,0,0,0,-0.000000,-0.0,0.000000,0.000000,-9.136467,0.000108
2020-04-24 12:00:00+00:00,-9.136467,-10.399756,-11.037855,-12.173192,-10.142192,-10.344540,-11.376537,-10.698591,-11.013714,-10.191359,...,0,0,0,1,-0.000000,-0.0,0.000000,0.000000,-9.878886,0.000051
2020-04-24 13:00:00+00:00,-9.878886,-9.136467,-10.399756,-10.835839,-10.731020,-9.506733,-9.125376,-10.832082,-9.459211,-9.805036,...,1,1,0,0,-9.878886,-0.0,6.351896,0.001266,-11.220871,0.000013
2020-04-24 14:00:00+00:00,-11.220871,-9.878886,-9.136467,-10.484218,-11.638972,-6.899771,-9.461127,-10.254837,-10.610571,-10.078742,...,1,1,0,0,-11.220871,-0.0,5.967658,0.003403,-9.154555,0.000106


In [7]:
df.columns

Index(['log_rv_lag_1h', 'log_rv_lag_2h', 'log_rv_lag_3h', 'log_rv_lag_6h',
       'log_rv_lag_12h', 'log_rv_lag_24h', 'log_rv_lag_48h', 'log_rv_lag_72h',
       'log_rv_lag_168h', 'log_rv_mean_3h', 'log_rv_mean_6h',
       'log_rv_mean_12h', 'log_rv_mean_24h', 'log_rv_mean_72h',
       'log_rv_mean_168h', 'log_rv_mean_720h', 'log_rv_std_24h',
       'log_rv_std_72h', 'log_rv_std_168h', 'log_rv_std_720h', 'log_rv_ewm_6h',
       'log_rv_ewm_12h', 'log_rv_ewm_24h', 'log_rv_ewm_72h', 'log_rv_ewm_168h',
       'log_rv_zscore_24h', 'log_rv_zscore_168h', 'ret_lag_1h', 'ret_lag_2h',
       'ret_lag_3h', 'ret_lag_6h', 'ret_lag_12h', 'ret_lag_24h',
       'abs_ret_lag_1h', 'abs_ret_lag_3h', 'abs_ret_lag_6h', 'abs_ret_lag_24h',
       'abs_ret_mean_6h', 'abs_ret_mean_24h', 'abs_ret_mean_168h',
       'hl_range_lag_1h', 'oc_range_lag_1h', 'hl_range_lag_24h',
       'oc_range_lag_24h', 'hl_range_lag_168h', 'oc_range_lag_168h',
       'hl_range_mean_24h', 'oc_range_mean_24h', 'hl_range_mean_168h',


In [4]:
def qlike(y_true_rv, y_pred_rv, eps=1e-12):
    y_true_rv = np.maximum(y_true_rv, eps)
    y_pred_rv = np.maximum(y_pred_rv, eps)

    return np.mean(
        y_true_rv / y_pred_rv - np.log(y_true_rv / y_pred_rv) - 1
    )


def regression_metrics(y_true_log, y_pred_log, eps=1e-12):
    y_true_log = np.asarray(y_true_log)
    y_pred_log = np.asarray(y_pred_log)

    y_true_rv = np.exp(y_true_log) - eps
    y_pred_rv = np.exp(y_pred_log) - eps

    return {
        "mse_log": np.mean((y_true_log - y_pred_log) ** 2),
        "mae_log": np.mean(np.abs(y_true_log - y_pred_log)),
        "mse_rv": np.mean((y_true_rv - y_pred_rv) ** 2),
        "mae_rv": np.mean(np.abs(y_true_rv - y_pred_rv)),
        "qlike": qlike(y_true_rv, y_pred_rv),
        "pearson": np.corrcoef(y_true_log, y_pred_log)[0, 1],
        "spearman": spearmanr(y_true_log, y_pred_log).correlation
    }

In [5]:
def make_walk_forward_splits(
    data,
    train_years=2,
    val_months=1,
    test_months=2,
    step_months=2,
    use_val=True
):
    start = data.index.min() + pd.DateOffset(years=train_years)
    end = data.index.max()

    splits = []
    split_start = start

    while True:
        train_start = split_start - pd.DateOffset(years=train_years)
        train_end = split_start

        if use_val:
            val_start = train_end
            val_end = val_start + pd.DateOffset(months=val_months)
            test_start = val_end
            test_end = test_start + pd.DateOffset(months=test_months)
        else:
            val_start = None
            val_end = None
            test_start = train_end
            test_end = test_start + pd.DateOffset(months=test_months)

        if test_end > end:
            break

        train_idx = (data.index >= train_start) & (data.index < train_end)
        test_idx = (data.index >= test_start) & (data.index < test_end)

        if use_val:
            val_idx = (data.index >= val_start) & (data.index < val_end)
        else:
            val_idx = None

        splits.append({
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_start": train_start,
            "train_end": train_end,
            "val_start": val_start,
            "val_end": val_end,
            "test_start": test_start,
            "test_end": test_end
        })

        split_start = split_start + pd.DateOffset(months=step_months)

    return splits

In [6]:
def run_walk_forward_with_val(
    model_fn,
    dataset,
    feature_cols,
    target_col="target_log_rv",
    output_path="predictions_walk_forward_val.parquet"
):
    splits = make_walk_forward_splits(
        dataset,
        train_years=2,
        val_months=1,
        test_months=2,
        step_months=2,
        use_val=True
    )

    preds = []
    metrics = []

    for i, split in enumerate(splits):
        train = dataset.loc[split["train_idx"]]
        val = dataset.loc[split["val_idx"]]
        test = dataset.loc[split["test_idx"]]

        model = model_fn()

        model.fit(
            train[feature_cols],
            train[target_col],
            eval_set=[(val[feature_cols], val[target_col])]
        )

        y_pred = model.predict(test[feature_cols])
        y_true = test[target_col].values

        fold_preds = pd.DataFrame({
            "datetime": test.index,
            "fold": i,
            "y_true_log_rv": y_true,
            "y_pred_log_rv": y_pred,
            "y_true_rv": np.exp(y_true) - 1e-12,
            "y_pred_rv": np.exp(y_pred) - 1e-12
        })

        preds.append(fold_preds)

        fold_metrics = regression_metrics(y_true, y_pred)
        fold_metrics["fold"] = i
        fold_metrics["test_start"] = split["test_start"]
        fold_metrics["test_end"] = split["test_end"]
        metrics.append(fold_metrics)

    preds = pd.concat(preds).set_index("datetime")
    metrics = pd.DataFrame(metrics)

    preds.to_parquet(output_path)

    return preds, metrics, metrics.drop(columns=["fold", "test_start", "test_end"]).mean()

In [10]:
y = df["target_log_rv"]
rv = np.exp(y)
df["target_log_rv_1h"] = y.shift(-1)
df["target_log_rv_6h"] = np.log(rv.shift(-1).rolling(6).mean().shift(-5))
df["target_log_rv_24h"] = np.log(rv.shift(-1).rolling(24).mean().shift(-23))
df["target_log_rv_168h"] = np.log(rv.shift(-1).rolling(168).mean().shift(-167))
df = df.dropna(subset=["target_log_rv_1h", "target_log_rv_6h", "target_log_rv_24h", "target_log_rv_168h"])

In [16]:
targets = ["target_log_rv_1h", "target_log_rv_6h", "target_log_rv_24h", "target_log_rv_168h"]
feature_cols = [c for c in df.columns if not c.startswith("target")]

all_preds = []
all_metrics = []

def run_optuna_lgbm(dataset, features, target, n_trials=30):
    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "mse",
            "boosting_type": "gbdt",
            "verbosity": -1,
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),
        }

        def lgb_model_fn():
            return lgb.LGBMRegressor(**params, n_estimators=500)

        _, _, avg_metrics = run_walk_forward_with_val(
            model_fn=lgb_model_fn,
            dataset=dataset,
            feature_cols=features,
            target_col=target,
            output_path=None)

        return avg_metrics["mse_log"]

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_trial.params

for target in targets:
    features = [c for c in feature_cols if not c.startswith("target")]
    
    print(f"{target}...")
    best_params = run_optuna_lgbm(df, features, target, n_trials=15)
    print(f"best for {target}: {best_params}")
    final_model = lgb.LGBMRegressor(
        **best_params,
        objective="regression",
        metric="mse",
        n_estimators=400)
    
    preds, metrics, avg_metrics = run_walk_forward_with_val(
        model_fn=lambda: final_model,
        dataset=df,
        feature_cols=features,
        target_col=target,
        output_path=f"preds_final_{target}.parquet"
    )
    preds = preds.reset_index()
    preds["target"] = target
    metrics["target"] = target
    all_preds.append(preds)
    all_metrics.append(metrics)
all_preds_df = pd.concat(all_preds).set_index("datetime")
all_metrics_df = pd.concat(all_metrics).reset_index(drop=True)

all_preds_df.to_parquet("all_preds_lgbm.parquet", index=True)
all_metrics_df.to_parquet("all_metrics_lgbm.parquet", index=False)

[I 2026-05-10 17:33:47,332] A new study created in memory with name: no-name-3273bb93-b626-4192-942a-aa5ab1871c9c


target_log_rv_1h...


[I 2026-05-10 17:34:45,990] Trial 0 finished with value: 0.6294172161187703 and parameters: {'num_leaves': 115, 'max_depth': 11, 'learning_rate': 0.005564026359032156, 'min_data_in_leaf': 382, 'feature_fraction': 0.7669071953396461, 'bagging_fraction': 0.8651164685619672, 'bagging_freq': 4, 'lambda_l1': 0.11261944515650289, 'lambda_l2': 0.0003288728181456169, 'min_gain_to_split': 0.02525969757155233}. Best is trial 0 with value: 0.6294172161187703.
[I 2026-05-10 17:35:59,980] Trial 1 finished with value: 0.5917317785302921 and parameters: {'num_leaves': 42, 'max_depth': 7, 'learning_rate': 0.006845343856612144, 'min_data_in_leaf': 173, 'feature_fraction': 0.8850289338327848, 'bagging_fraction': 0.8654543156747561, 'bagging_freq': 8, 'lambda_l1': 0.009478067451543377, 'lambda_l2': 0.00036033009986360134, 'min_gain_to_split': 0.9448045010895206}. Best is trial 1 with value: 0.5917317785302921.
[I 2026-05-10 17:36:36,084] Trial 2 finished with value: 1.0715623658472868 and parameters: {'n

best for target_log_rv_1h: {'num_leaves': 218, 'max_depth': 4, 'learning_rate': 0.016847943117968348, 'min_data_in_leaf': 117, 'feature_fraction': 0.8943777813971469, 'bagging_fraction': 0.9193959165687134, 'bagging_freq': 10, 'lambda_l1': 2.024549413160678, 'lambda_l2': 0.005020054717513629, 'min_gain_to_split': 0.962605602549852}


[I 2026-05-10 17:45:23,796] A new study created in memory with name: no-name-86356ed7-6699-44ca-8715-6e9a74e80187


...

best for target_log_rv_24h: {'num_leaves': 58, 'max_depth': 6, 'learning_rate': 0.008720375907822526, 'min_data_in_leaf': 298, 'feature_fraction': 0.7810498712843325, 'bagging_fraction': 0.6879364762778514, 'bagging_freq': 7, 'lambda_l1': 1.095251864961605e-06, 'lambda_l2': 4.808180666176153e-05, 'min_gain_to_split': 0.3899061499411054}


[I 2026-05-10 18:05:33,876] A new study created in memory with name: no-name-d722ad87-c405-4f65-b211-f9a9f47cb4f3


target_log_rv_168h...


[I 2026-05-10 18:07:20,516] Trial 0 finished with value: 0.5252992221414794 and parameters: {'num_leaves': 221, 'max_depth': 11, 'learning_rate': 0.0021885826795912935, 'min_data_in_leaf': 190, 'feature_fraction': 0.750968177200585, 'bagging_fraction': 0.9350562569022111, 'bagging_freq': 7, 'lambda_l1': 1.408184810730704e-05, 'lambda_l2': 1.6354781319449304e-05, 'min_gain_to_split': 0.10415176994070097}. Best is trial 0 with value: 0.5252992221414794.
[I 2026-05-10 18:07:41,332] Trial 1 finished with value: 0.5469288916281158 and parameters: {'num_leaves': 210, 'max_depth': 3, 'learning_rate': 0.031706988075410794, 'min_data_in_leaf': 472, 'feature_fraction': 0.8317972513487579, 'bagging_fraction': 0.8611904792696788, 'bagging_freq': 10, 'lambda_l1': 3.355355437185691e-07, 'lambda_l2': 0.5044682577218244, 'min_gain_to_split': 0.12274605932107141}. Best is trial 0 with value: 0.5252992221414794.
[I 2026-05-10 18:08:25,108] Trial 2 finished with value: 0.4683015245906278 and parameters: 

best for target_log_rv_168h: {'num_leaves': 30, 'max_depth': 3, 'learning_rate': 0.004393936214277764, 'min_data_in_leaf': 401, 'feature_fraction': 0.653348630636249, 'bagging_fraction': 0.8163723526286084, 'bagging_freq': 1, 'lambda_l1': 0.021733711129268862, 'lambda_l2': 9.783868737483294, 'min_gain_to_split': 0.24568496453918418}
